In [ ]:
import math
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def compute_inter(box1, box2):
    x1 = torch.max(box1[0], box2[0])
    y1 = torch.max(box1[1], box2[1])
    x2 = torch.min(box1[2], box2[2])
    y2 = torch.min(box1[3], box2[3])

    return torch.clamp(x2 - x1, min=0) * torch.clamp(y2 - y1, min=0)   

def compute_box_area(box):
    return (box[2] - box[0]) * (box[3] - box[1])

def compute_iou(box1, box2):
    inter = compute_inter(box1, box2)
    area1 = compute_box_area(box1)
    area2 = compute_box_area(box2)
    union = area1 + area2 - inter

    return inter / (union + 1e-7)

def compute_giou(box1, box2):
    iou = compute_iou(box1, box2)
    
    cx1 = torch.min(box1[0], box2[0])
    cy1 = torch.min(box1[1], box2[1])
    cx2 = torch.max(box1[2], box2[2])
    cy2 = torch.max(box1[3], box2[3])
    c_area = (cx2 - cx1) * (cy2 - cy1)

    area1 = compute_box_area(box1)
    area2 = compute_box_area(box2)

    inter = compute_inter(box1, box2)
    union = area1 + area2 - inter

    giou = iou - (c_area - union) / (c_area + 1e-7)
    return giou

def compute_ciou(box1, box2):
    iou = compute_iou(box1, box2)

    dx1 = (box1[0] + box1[2]) / 2
    dy1 = (box1[1] + box1[3]) / 2
    dx2 = (box2[0] + box2[2]) / 2
    dy2 = (box2[1] + box2[3]) / 2
    d2 = (dx1 - dx2) ** 2 + (dy1 - dy2) ** 2

    cx1 = torch.min(box1[0], box2[0])
    cy1 = torch.min(box1[1], box2[1])
    cx2 = torch.max(box1[2], box2[2])
    cy2 = torch.max(box1[3], box2[3])
    c2 = (cx2 - cx1) ** 2 + (cy2 - cy1) ** 2

    w1 = box1[2] - box1[0]
    h1 = box1[3] - box1[1]
    w2 = box2[2] - box2[0]
    h2 = box2[3] - box2[1]

    v = (4 / math.pi ** 2) * (torch.atan(w2 / (h2 + 1e-7)) - torch.atan(w1 / (h1 + 1e-7))) ** 2

    with torch.no_grad():
        alpha = v / (1 - iou + v + 1e-7)

    ciou = iou - d2 / (c2 + 1e-7) - alpha * v
    return ciou




# -- 시나리오 비교 --
# GT는 (100, 100)~(200, 200)의 100x100 정사각형, 중심은 (150, 150)으로 고정
gt = torch.tensor([100., 100., 200., 200.]) # 정답 박스 (x1, y1, x2, y2)

# 6가지 시나리오: (시각화용 영어 제목, pred 박스, 콘솔 출력용 한국어 설명)
# title은 matplotlib에 한글 폰트가 없을 때 깨지므로 영어, desc는 콘솔 출력용이라 한국어 유지
scenarios = [
    ("Nearly accurate",
     torch.tensor([102., 98., 202., 198.]),
     "GT와 거의 일치 -> IoU 1에 근접"),

    ("Partial overlap",
     torch.tensor([120., 110., 220., 210.]),
     "대각선으로 약간 이동 -> 일반적인 학습 중 상황"),

    ("Adjacent (IoU=0)",
     torch.tensor([220., 100., 320., 200.]),
     "GT 바로 옆 (IoU=0) -> GIoU는 약한 음수로 거리 신호 전달"),

    ("Far apart (IoU=0)",
     torch.tensor([400., 400., 500., 500.]),
     "GT에서 멀리 (IoU=0) -> GIoU가 더 작음 (GIoU의 거리 민감도)"),

    ("Pred contains GT",
     torch.tensor([50., 50., 250., 250.]),
     "pred가 GT를 완전히 감쌈 -> 외접 영역 C = pred, GIoU=IoU"),

    ("Same center, diff aspect",
     torch.tensor([100., 130., 200., 170.]),
     "중심점은 같으나 가로로 납작 -> CIoU의 종횡비 패널티가 드러남"),
]

print(f"\nGT: {gt.tolist()} (100x100 정사각형, 중심 (150, 150))")
for title, pred, desc in scenarios: # 시나리오마다 IoU/GIoU/CIoU 출력
    iou = compute_iou(pred, gt).item()
    giou = compute_giou(pred, gt).item()
    ciou = compute_ciou(pred, gt).item()
    print(f"\n[{title}] {desc}")
    print(f"  Pred: {pred.tolist()}")
    print(f"  IoU={iou:.4f}, GIoU={giou:.4f}, CIoU={ciou:.4f}")


# -- 시각화 --
fig, axes = plt.subplots(2, 3, figsize=(15, 10)) # 6개 시나리오를 2x3 그리드로 배치
axes = axes.flatten() # 2x3을 1차원으로 펴서 zip으로 순회


for ax, (title, pred, _) in zip(axes, scenarios): # 시나리오마다 박스 그리기
    ax.set_xlim(0, 550)
    ax.set_ylim(550, 0) # y축 뒤집기 (이미지 좌표계는 위가 0)
    ax.set_aspect('equal') # 가로세로 비율 동일하게
    ax.set_title(title, fontsize=11)


    # GT Box (녹색)
    rect_gt = patches.Rectangle(
        (gt[0], gt[1]), gt[2]-gt[0], gt[3]-gt[1], # (좌상단 좌표), 너비, 높이
        linewidth=2, edgecolor='green', facecolor='green', alpha=0.3,
        label='GT')
    ax.add_patch(rect_gt) # 그래프에 사각형 추가


    # Pred Box (빨간)
    rect_pred = patches.Rectangle(
        (pred[0], pred[1]), pred[2]-pred[0], pred[3]-pred[1],
        linewidth=2, edgecolor='red', facecolor='red', alpha=0.3,
        label='Pred')
    ax.add_patch(rect_pred)


    iou_val = compute_iou(pred, gt).item()
    giou_val = compute_giou(pred, gt).item()
    ciou_val = compute_ciou(pred, gt).item()
    ax.text(275, 520, # 각 subplot 하단 중앙에 세 지표 모두 표시
            f"IoU={iou_val:.3f}  GIoU={giou_val:.3f}  CIoU={ciou_val:.3f}",
            fontsize=9, ha='center')
    ax.legend(loc='upper right', fontsize=8)


plt.tight_layout()
plt.savefig('iou_comparison.png', dpi=100) # 결과를 이미지 파일로 저장
print("\n시각화 저장: iou_comparison.png")



GT: [100.0, 100.0, 200.0, 200.0] (100x100 정사각형, 중심 (150, 150))

[Nearly accurate] GT와 거의 일치 -> IoU 1에 근접
  Pred: [102.0, 98.0, 202.0, 198.0]
  IoU=0.9238, GIoU=0.9230, CIoU=0.9234

[Partial overlap] 대각선으로 약간 이동 -> 일반적인 학습 중 상황
  Pred: [120.0, 110.0, 220.0, 210.0]
  IoU=0.5625, GIoU=0.5322, CIoU=0.5436

[Adjacent (IoU=0)] GT 바로 옆 (IoU=0) -> GIoU는 약한 음수로 거리 신호 전달
  Pred: [220.0, 100.0, 320.0, 200.0]
  IoU=0.0000, GIoU=-0.0909, CIoU=-0.2466

[Far apart (IoU=0)] GT에서 멀리 (IoU=0) -> GIoU가 더 작음 (GIoU의 거리 민감도)
  Pred: [400.0, 400.0, 500.0, 500.0]
  IoU=0.0000, GIoU=-0.8750, CIoU=-0.5625

[Pred contains GT] pred가 GT를 완전히 감쌈 -> 외접 영역 C = pred, GIoU=IoU
  Pred: [50.0, 50.0, 250.0, 250.0]
  IoU=0.2500, GIoU=0.2500, CIoU=0.2500

[Same center, diff aspect] 중심점은 같으나 가로로 납작 -> CIoU의 종횡비 패널티가 드러남
  Pred: [100.0, 130.0, 200.0, 170.0]
  IoU=0.4000, GIoU=0.4000, CIoU=0.3934

시각화 저장: iou_comparison.png

 실습 1 완료!


In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def compute_iou_np(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    iou = inter / (area1 + area2 - inter + 1e-7)
    return iou

